# Stage 1 — Label Fixing
**Goal:** Merge attack families, drop unusable classes, encode to numbers.

---

## What happens to each label

| Original Label | Count | Action | Final Label |
|---|---|---|---|
| BENIGN | 2,271,320 | ✅ Keep as-is | `0 — BENIGN` |
| DoS Hulk | 230,124 | 🔀 Merge → DoS | `1 — DoS` |
| DoS GoldenEye | 10,293 | 🔀 Merge → DoS | `1 — DoS` |
| DoS slowloris | 5,796 | 🔀 Merge → DoS | `1 — DoS` |
| DoS Slowhttptest | 5,499 | 🔀 Merge → DoS | `1 — DoS` |
| DDoS | 128,025 | ✅ Keep as-is | `2 — DDoS` |
| PortScan | 158,804 | ✅ Keep as-is | `3 — PortScan` |
| FTP-Patator | 7,935 | 🔀 Merge → BruteForce | `4 — BruteForce` |
| SSH-Patator | 5,897 | 🔀 Merge → BruteForce | `4 — BruteForce` |
| Web Attack  Brute Force | 1,507 | 🔀 Merge → WebAttack | `5 — WebAttack` |
| Web Attack  XSS | 652 | 🔀 Merge → WebAttack | `5 — WebAttack` |
| Web Attack  Sql Injection | 21 | 🔀 Merge → WebAttack | `5 — WebAttack` |
| Bot | 1,956 | ✅ Keep as-is | `6 — Bot` |
| Infiltration | 36 | ❌ Dropped — too few rows | — |
| Heartbleed | 11 | ❌ Dropped — too few rows | — |

---

## Final encoding

| Number | Class | Total rows (approx) |
|---|---|---|
| 0 | BENIGN | 2,271,320 |
| 1 | DoS | 251,712 |
| 2 | DDoS | 128,025 |
| 3 | PortScan | 158,804 |
| 4 | BruteForce | 13,832 |
| 5 | WebAttack | 2,180 |
| 6 | Bot | 1,956 |
| — | ~~Infiltration~~ | ~~36~~ dropped |
| — | ~~Heartbleed~~ | ~~11~~ dropped |

In [1]:
import pandas as pd
import numpy as np

CSV_PATH = r'D:\Computer Engineering 2022\4-CE.Z Forth Year\second-semester-4.2\project\ML-NIDS-Android\03-dataset\02-processed\combined_clean.csv'

print('Loading CSV...')
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
print(f'Loaded shape: {df.shape}')
print('\nOriginal label distribution:')
print(df['Label'].value_counts())

Loading CSV...
Loaded shape: (2827876, 85)

Original label distribution:
Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack  Brute Force         1507
Web Attack  XSS                  652
Infiltration                       36
Web Attack  Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


# Diagnostic — Find Exact Label Bytes
Run this BEFORE the label mapping cell to see exactly what characters are in the problematic labels.

In [2]:
# ── DIAGNOSTIC: See exact bytes of every unique label ────────────────────────
print('=== Exact label strings and their bytes ===\n')
for label in df['Label'].unique():
    if 'Web' in str(label) or 'web' in str(label):
        print(f'Label  : {repr(label)}')
        print(f'Bytes  : {label.encode("unicode_escape")}')
        print()

=== Exact label strings and their bytes ===

Label  : 'Web Attack \x96 Brute Force'
Bytes  : b'Web Attack \\x96 Brute Force'

Label  : 'Web Attack \x96 XSS'
Bytes  : b'Web Attack \\x96 XSS'

Label  : 'Web Attack \x96 Sql Injection'
Bytes  : b'Web Attack \\x96 Sql Injection'



In [3]:
# ── FIX: Normalize labels before mapping ─────────────────────────────────────
import re

def normalize_label(s):
    """Remove hidden chars, normalize dashes, collapse spaces, strip edges."""
    if not isinstance(s, str):
        return s
    # Replace non-breaking space (\xa0) and other unicode spaces with regular space
    s = s.replace('\xa0', ' ')
    # Replace en-dash (–) and em-dash (—) with regular hyphen
    s = s.replace('\u2013', '-').replace('\u2014', '-')
    # Collapse multiple spaces into one
    s = re.sub(r' +', ' ', s)
    # Strip leading/trailing whitespace
    s = s.strip()
    return s

# Apply normalization first
df['Label'] = df['Label'].apply(normalize_label)

# Print all unique labels after normalization so you can verify
print('All unique labels after normalization:')
for label in sorted(df['Label'].dropna().unique()):
    print(f'  {repr(label)}')

All unique labels after normalization:
  'BENIGN'
  'Bot'
  'DDoS'
  'DoS GoldenEye'
  'DoS Hulk'
  'DoS Slowhttptest'
  'DoS slowloris'
  'FTP-Patator'
  'Heartbleed'
  'Infiltration'
  'PortScan'
  'SSH-Patator'
  'Web Attack \x96 Brute Force'
  'Web Attack \x96 Sql Injection'
  'Web Attack \x96 XSS'


In [4]:
# ── SAFE MAP: Build map from actual labels found in your data ─────────────────
# After running the cell above, copy the exact repr() strings shown
# and paste them as keys below. This guarantees a 100% match.

# Check what Web Attack labels actually look like now
web_labels = [l for l in df['Label'].dropna().unique() if 'Web' in str(l) or 'web' in str(l)]
print('Web Attack labels found in your data:')
for l in web_labels:
    print(f'  {repr(l)}')

print('\nCopy these exactly into your LABEL_MAP below.')

Web Attack labels found in your data:
  'Web Attack \x96 Brute Force'
  'Web Attack \x96 XSS'
  'Web Attack \x96 Sql Injection'

Copy these exactly into your LABEL_MAP below.


In [5]:
# ── FIX: Normalize labels before mapping ─────────────────────────────────────
import re

def normalize_label(s):
    if not isinstance(s, str):
        return s
    s = s.replace('\xa0', ' ')      # non-breaking space
    s = s.replace('\x96', ' ')      # Windows-1252 en-dash  ← YOUR CASE
    s = s.replace('\u2013', ' ')    # unicode en-dash
    s = s.replace('\u2014', ' ')    # em-dash
    s = re.sub(r' +', ' ', s)       # collapse multiple spaces
    s = s.strip()
    return s

df['Label'] = df['Label'].apply(normalize_label)

# Verify
print('Web Attack labels after fix:')
for l in sorted(df['Label'].dropna().unique()):
    if 'Web' in str(l):
        print(f'  {repr(l)}')

Web Attack labels after fix:
  'Web Attack Brute Force'
  'Web Attack Sql Injection'
  'Web Attack XSS'


In [6]:
# ── Step 1: Strip whitespace from all label strings ──────────────────────────
df['Label'] = df['Label'].str.strip()

# ── Step 2: Drop rows where Label is NaN ─────────────────────────────────────
before = len(df)
df.dropna(subset=['Label'], inplace=True)
print(f'Dropped {before - len(df)} NaN label rows')

# ── Step 3: Drop Infiltration and Heartbleed (too few samples) ───────────────
DROP_LABELS = ['Infiltration', 'Heartbleed']
before = len(df)
df = df[~df['Label'].isin(DROP_LABELS)]
print(f'Dropped {before - len(df)} rows (Infiltration + Heartbleed)')

# ── Step 4: Merge attack families ────────────────────────────────────────────
# ── UPDATED LABEL_MAP using normalized strings ────────────────────────────────
# These keys match the output of normalize_label() above

LABEL_MAP = {
    'BENIGN'                     : 'BENIGN',

    # DoS family
    'DoS Hulk'                   : 'DoS',
    'DoS GoldenEye'              : 'DoS',
    'DoS slowloris'              : 'DoS',
    'DoS Slowhttptest'           : 'DoS',

    'DDoS'                       : 'DDoS',
    'PortScan'                   : 'PortScan',

    # BruteForce family
    'FTP-Patator'                : 'BruteForce',
    'SSH-Patator'                : 'BruteForce',

    # WebAttack family — single space after normalization
    'Web Attack Brute Force'     : 'WebAttack',
    'Web Attack XSS'             : 'WebAttack',
    'Web Attack Sql Injection'   : 'WebAttack',

    'Bot'                        : 'Bot',
}

df['Label'] = df['Label'].map(LABEL_MAP)

unmapped = df['Label'].isna().sum()
if unmapped > 0:
    print(f'Still unmapped: {unmapped} rows')
    print('Run the diagnostic cell above to find remaining issues.')
else:
    print('All labels mapped successfully — 0 unmapped rows')

print('\nLabel distribution after mapping:')
print(df['Label'].value_counts())

Dropped 0 NaN label rows
Dropped 47 rows (Infiltration + Heartbleed)
All labels mapped successfully — 0 unmapped rows

Label distribution after mapping:
Label
BENIGN        2271320
DoS            251712
PortScan       158804
DDoS           128025
BruteForce      13832
WebAttack        2180
Bot              1956
Name: count, dtype: int64


In [7]:
# ── Step 5: Encode labels to integers ────────────────────────────────────────
ENCODE_MAP = {
    'BENIGN'     : 0,
    'DoS'        : 1,
    'DDoS'       : 2,
    'PortScan'   : 3,
    'BruteForce' : 4,
    'WebAttack'  : 5,
    'Bot'        : 6,
}

df['Label'] = df['Label'].map(ENCODE_MAP)

print('Final encoded label distribution:')
label_names = {v: k for k, v in ENCODE_MAP.items()}
counts = df['Label'].value_counts().sort_index()
for code, count in counts.items():
    print(f'  {code} — {label_names[code]:<12} : {count:,}')

print(f'\nFinal shape: {df.shape}')

Final encoded label distribution:
  0 — BENIGN       : 2,271,320
  1 — DoS          : 251,712
  2 — DDoS         : 128,025
  3 — PortScan     : 158,804
  4 — BruteForce   : 13,832
  5 — WebAttack    : 2,180
  6 — Bot          : 1,956

Final shape: (2827829, 85)


In [ ]:
# ── Step 6: Save labeled dataset ─────────────────────────────────────────────
import os

# ─── Destination folder ─────────────────────────────────────────────
DEST_FOLDER = r"D:\Computer Engineering 2022\4-CE.Z Forth Year\second-semester-4.2\project\ML-NIDS-Android\03-dataset\02-processed"

# Make sure folder exists
os.makedirs(DEST_FOLDER, exist_ok=True)

# Build full output path
OUT_PATH = os.path.join(DEST_FOLDER, "02_combined_labeled.csv")

# Save
df.to_csv(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")



Saved → D:\Computer Engineering 2022\4-CE.Z Forth Year\second-semester-4.2\project\ML-NIDS-Android\03-dataset\02-processed\02_combined_labeled.csv


In [3]:
import pandas as pd
import numpy as np

CSV_PATH = r'D:\Computer Engineering 2022\4-CE.Z Forth Year\second-semester-4.2\project\ML-NIDS-Android\03-dataset\02-processed\02_combined_labeled.csv'

print('Loading CSV...')
df = pd.read_csv(CSV_PATH)

# Strip whitespace from column names
df.columns = df.columns.str.strip()

print(f'Loaded shape: {df.shape}')

# Show all column names
print("\nAll columns:")
print(df.columns.tolist())

print('\nOriginal label distribution:')
print(df['Label'].value_counts())
# Count duplicate rows
duplicate_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicate_count}")


Loading CSV...
Loaded shape: (2827829, 85)

All columns:
['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'F